In [11]:
import pandas as pd
import numpy as np
import config
import utils
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

In [9]:
#random state 
rs = 7919

# Load data
df_com = pd.read_csv("../data/data_communities.csv", index_col=[0])
df = pd.read_csv(config.FEATS_PATH, header=None)
#df_labels = pd.read_csv(config.LABELS_PATH)
df = df.merge(df_com, left_on=0, right_on="txId").drop(["txId"], axis=1)
df.loc[:, "target"] = df[1]

/tmp/ipykernel_4756/2952387645.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.loc[:, "target"] = df[1]


In [3]:
feats_auc = pd.DataFrame(columns=["feature", "auc"])
for feat in range(1, 167):
    auc = utils.pred_power(feat, "label", df.loc[~df.loc[:, "label"].isnull()])
    feats_auc.loc[len(feats_auc)] = [feat, auc]

In [4]:
# let's only keep the most powerful features and remove the correlated ones
# also, add the graph features
features_to_keep = utils.filter_corr(df.loc[~df.loc[:, "label"].isnull(), :], feats_auc, thr=0.5)
features_to_keep.extend(["n_nodes", "n_illicit", "n_licit", "n_br"])

In [7]:
# Create X and y
X = df.loc[df.loc[:, "label"].notnull(), features_to_keep]
y = df.loc[df.loc[:, "label"].notnull(), "label"]
X.columns = X.columns.astype(str)

In [12]:
# full grid search CV with a decision tree
param_grid_tree = {
    "max_depth": [3, 5, 7],
    "min_samples_split": [5, 10, 20]
}

grid_search_tree = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=rs), # prime number
    param_grid=param_grid_tree,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=rs), 
    scoring="roc_auc"
)
grid_search_tree.fit(X, y)

print(grid_search_tree.best_score_)

0.9755978691098715


In [14]:
param_grid_xgb = {
    "max_depth": [2, 4, 6],
    "learning_rate": [0.01, 0.1, 0.3],
    "n_estimators": [20, 50],
    "subsample": [0.5, 1]
}

grid_search_xgb = GridSearchCV(
    estimator=XGBClassifier(random_state=rs, eval_metric="logloss"),
    param_grid=param_grid_xgb,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=rs),
    scoring="roc_auc",
    n_jobs=-1
)
grid_search_xgb.fit(X, y)
print(grid_search_xgb.best_score_)

0.9968096683061815


In [18]:
grid_search_xgb.best_estimator_.predict_proba(X)

array([[9.9995208e-01, 4.7898942e-05],
       [9.9992508e-01, 7.4914060e-05],
       [9.9694526e-01, 3.0547599e-03],
       ...,
       [1.7866910e-02, 9.8213309e-01],
       [9.9984223e-01, 1.5776973e-04],
       [3.0950248e-02, 9.6904975e-01]], shape=(46564, 2), dtype=float32)